<a href="https://colab.research.google.com/github/shiiid/ShidDatamining2026/blob/main/hasilprediksi_157.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*   Muhammad Shidqi Al Ghiffari
*   2304020157



Langkah pertama yang harus dilakukan adalah mengimpor semua library Python yang akan kita gunakan selama proses analisis, pemodelan, hingga evaluasi.

* pandas: Digunakan untuk membaca, memanipulasi, dan mengolah dataset dalam bentuk dataframe.

* numpy: Digunakan untuk operasi angka dan komputasi matematis.

* train_test_split: Digunakan untuk membagi data pelatihan menjadi dua bagian: data latih (untuk melatih model) dan data validasi (untuk menguji model sementara sebelum ke data uji asli).

* StandardScaler: Digunakan untuk menormalkan/melakukan scaling pada fitur-fitur kimiawi agar memiliki skala yang seragam.

* accuracy_score, classification_report, confusion_matrix: Metrik yang digunakan untuk mengevaluasi seberapa baik performa model klasifikasi yang kita buat.

* Logistic Regression, KNeighbors Classifier, Decision Tree, ClassifierTiga algoritma model klasifikasi yang akan kita bandingkan.

In [1]:
import pandas as pd
import numpy as np

# Library untuk pemrosesan data & evaluasi
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Library untuk model klasifikasi
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

Dataset training (data_training.csv) dibaca terlebih dahulu karena data ini digunakan untuk melatih model. Data training memiliki kolom fitur kimiawi dan satu kolom target yaitu quality. Selanjutnya, kita juga membaca data_testing.csv yang nantinya akan kita prediksi nilai quality-nya.

In [3]:
train = pd.read_csv("data_training.csv")
test = pd.read_csv("data_testing.csv")

display(train.head())
display(test.head())

print("Ukuran data training:", train.shape)
print("Ukuran data testing:", test.shape)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.3,0.740,0.08,1.7,0.094,10.0,45.0,0.99576,3.24,0.50,9.8,5,1366
1,8.1,0.575,0.22,2.1,0.077,12.0,65.0,0.99670,3.29,0.51,9.2,5,103
2,10.1,0.430,0.40,2.6,0.092,13.0,52.0,0.99834,3.22,0.64,10.0,7,942
3,12.9,0.500,0.55,2.8,0.072,7.0,24.0,1.00012,3.09,0.68,10.9,6,811
4,8.4,0.360,0.32,2.2,0.081,32.0,79.0,0.99640,3.30,0.72,11.0,6,918


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,Id
0,6.8,0.61,0.04,1.5,0.057,5.0,10.0,0.99525,3.42,0.60,9.500000,222
1,6.9,0.84,0.21,4.1,0.074,16.0,65.0,0.99842,3.53,0.72,9.233333,1514
2,7.0,0.58,0.12,1.9,0.091,34.0,124.0,0.99560,3.44,0.48,10.500000,417
3,7.8,0.48,0.68,1.7,0.415,14.0,32.0,0.99656,3.09,1.06,9.100000,754
4,12.5,0.60,0.49,4.3,0.100,5.0,14.0,1.00100,3.25,0.74,11.900000,516


Ukuran data training: (857, 13)
Ukuran data testing: (286, 12)


Kita perlu memastikan bahwa dataset bersih dan tidak ada nilai yang hilang (missing values). Jika ada data yang kosong, hal ini bisa membuat model error saat dilatih.

In [4]:
# Mengecek apakah ada nilai yang hilang/kosong pada data training
print("Jumlah Missing Values pada Data Training:")
print(df_train.isnull().sum())

# Mengecek nilai hilang pada data testing
print("\nJumlah Missing Values pada Data Testing:")
print(df_test.isnull().sum())

Jumlah Missing Values pada Data Training:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64

Jumlah Missing Values pada Data Testing:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
Id                      0
dtype: int64


Hasil dari kode di atas menunjukkan angka 0 untuk semua kolom, itu berarti dataset sudah bersih dan tidak memerlukan tahap imputasi (pengisian nilai) atau penghapusan baris/kolom.

Model memerlukan dua komponen utama untuk belajar: X (Fitur/Variabel Independen) dan y (Target/Variabel Dependen).

* Pada data training, kolom quality adalah target yang harus diprediksi, sehingga kita pisahkan ke dalam variabel y.

* Kolom Id hanyalah nomor identifikasi dan tidak memiliki hubungan kimiawi dengan kualitas anggur, sehingga harus kita hapus agar tidak membingungkan model.

In [5]:
# Memisahkan fitur (X) dan target (y) pada data training
X = df_train.drop(columns=['quality', 'Id'])
y = df_train['quality']

# Untuk data testing, kita hanya perlu membuang kolom Id untuk diprediksi nanti
X_test_final = df_test.drop(columns=['Id'])

Membagi Data Training & Normalisasi (Feature Scaling)

Kita akan membagi X dan y menjadi data latih (80%) dan data validasi (20%). Setelah itu, fitur-fitur kimiawi seperti sulfur dioxide (yang angkanya bisa puluhan) dan chlorides (yang angkanya koma desimal kecil) perlu dinormalisasi menggunakan StandardScaler agar skalanya seragam. Jika tidak, model seperti KNN akan bias terhadap fitur yang angkanya besar.

In [6]:
# Membagi data menjadi data latih dan data validasi
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Menginisialisasi scaler
scaler = StandardScaler()

# Melakukan fitting dan scaling pada data latih, lalu scaling pada data validasi dan data uji
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_final_scaled = scaler.transform(X_test_final)

Kita akan melatih tiga model berbeda: Logistic Regression, K-Nearest Neighbors (KNN), dan Decision Tree menggunakan X_train_scaled. Setelah model belajar, kita uji kemampuannya menggunakan X_val_scaled dan kita bandingkan hasil prediksinya dengan kunci jawaban (y_val) menggunakan metrik akurasi.

In [7]:
# Mendefinisikan ketiga model
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

# Melatih dan mengevaluasi setiap model secara bergantian
best_model_name = ""
best_accuracy = 0
best_model = None

for name, model in models.items():
    # 1. Melatih model
    model.fit(X_train_scaled, y_train)

    # 2. Memprediksi data validasi
    y_pred = model.predict(X_val_scaled)

    # 3. Menghitung akurasi
    acc = accuracy_score(y_val, y_pred)
    print(f"=== {name} ===")
    print(f"Akurasi: {acc * 100:.2f}%")

    # Mencari model dengan performa terbaik
    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name
        best_model = model

print(f"\nModel terbaik yang akan digunakan adalah: {best_model_name}")

=== Logistic Regression ===
Akurasi: 59.88%
=== K-Nearest Neighbors (KNN) ===
Akurasi: 49.42%
=== Decision Tree ===
Akurasi: 46.51%

Model terbaik yang akan digunakan adalah: Logistic Regression


Dari ketiga model tersebut, Anda akan mendapati bahwa Logistic Regression umumnya memberikan akurasi tertinggi pada klasifikasi awal untuk dataset ini. Mari kita lihat evaluasi detail untuk model terbaik menggunakan Classification Report dan Confusion Matrix.

In [8]:
print(f"Evaluasi Detail untuk {best_model_name}:")
y_pred_best = best_model.predict(X_val_scaled)

print("\nClassification Report:")
print(classification_report(y_val, y_pred_best, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_best))

Evaluasi Detail untuk Logistic Regression:

Classification Report:
              precision    recall  f1-score   support

           4       0.00      0.00      0.00         3
           5       0.62      0.82      0.71        67
           6       0.62      0.54      0.58        78
           7       0.40      0.29      0.33        21
           8       0.00      0.00      0.00         3

    accuracy                           0.60       172
   macro avg       0.33      0.33      0.32       172
weighted avg       0.57      0.60      0.58       172

Confusion Matrix:
[[ 0  3  0  0  0]
 [ 0 55 12  0  0]
 [ 0 29 42  6  1]
 [ 0  1 14  6  0]
 [ 0  0  0  3  0]]


Melatih Ulang Model Terbaik dengan Semua Data Training

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

if best_model_name == "Logistic Regression":
    final_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

elif best_model_name == "K-Nearest Neighbors (KNN)":
    final_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])

else: # Decision Tree
    final_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=42))
    ])

final_model.fit(X, y)

print("Model final berhasil dilatih menggunakan seluruh data training.")

Model final berhasil dilatih menggunakan seluruh data training.


In [12]:
# Memprediksi kualitas pada data testing yang sudah di-scaling
test_predictions = best_model.predict(X_test_final_scaled)

# Membuat DataFrame baru berisi Id dari data_testing.csv dan hasil prediksi quality
submission = pd.DataFrame({
    'Id': df_test['Id'],
    'quality': test_predictions
})
submission

,Id,quality
0,222,5
1,1514,5
2,417,5
3,754,5
4,516,6
...,...,...
281,1147,7
282,296,5
283,170,5
284,1439,6


Secara keseluruhan, proyek ini berhasil membangun model klasifikasi untuk memprediksi kualitas anggur berdasarkan komposisi kimiawinya melalui alur machine learning yang sistematis. Prosesnya diawali dengan tahap persiapan data yang mencakup pengecekan nilai kosong dan penerapan normalisasi menggunakan StandardScaler, sebuah langkah krusial agar model dapat belajar dengan seimbang dan tidak bias terhadap fitur-fitur yang memiliki rentang angka besar. Setelah data diproses, tiga algoritma berbeda, yakni Logistic Regression, K-Nearest Neighbors (KNN), dan Decision Tree dilatih dan dievaluasi kinerjanya untuk menentukan model dengan akurasi paling tinggi. Pada akhirnya, model dengan performa terbaik tersebut diaplikasikan untuk memprediksi label kualitas pada data uji